# Temporal RVQ-VAE cho video skeleton

Pipeline chỉ dùng ViTPose: temporal Conv1D encoder 63 bước → residual EMA quantization → temporal decoder → joint-token grammar 1–4 gram theo temporal pyramid, 1D CNN và BiGRU. Mỗi fold được tách theo trẻ và tối đa hai fold chạy song song trên hai GPU.

In [ ]:
import os
from pathlib import Path

IS_COLAB = "COLAB_RELEASE_TAG" in os.environ
DEFAULT_DATA_ROOT = "/content/drive/MyDrive/Fidgety/data" if IS_COLAB else "/kaggle/input/datasets/xndung/fidgety/data"
DEFAULT_SOURCE_DIR = "/content/Fidgety" if IS_COLAB else "/kaggle/working/Fidgety"
DEFAULT_OUTPUT_DIR = "/content/drive/MyDrive/Fidgety/results" if IS_COLAB else "/kaggle/working"
DATA_ROOT = Path(os.environ.get("FIDGETY_DATA_ROOT", DEFAULT_DATA_ROOT))
REPO_URL = "https://github.com/xn-dung/Fidgety.git"
SOURCE_DIR = Path(os.environ.get("FIDGETY_SOURCE_DIR", DEFAULT_SOURCE_DIR))
UPLOADED_PROJECT_DIR = None
RUN_MODE = os.environ.get("FIDGETY_RUN_MODE", "smoke")
SEED = 42
GPU_WORKERS = 2
CPU_THREADS_PER_WORKER = 2
OUTPUT_DIR = Path(os.environ.get("FIDGETY_OUTPUT_ROOT", DEFAULT_OUTPUT_DIR)) / f"temporal_rvq_grammar_{RUN_MODE}_v3"

if RUN_MODE == "smoke":
    FOLDS = [1]
    RVQ_EPOCHS = 8
    CLASSIFIER_EPOCHS = 15
    RVQ_PATIENCE = 4
    CLASSIFIER_PATIENCE = 5
elif RUN_MODE == "full":
    FOLDS = list(range(1, 10))
    RVQ_EPOCHS = 120
    CLASSIFIER_EPOCHS = 120
    RVQ_PATIENCE = 12
    CLASSIFIER_PATIENCE = 15
else:
    raise ValueError("RUN_MODE must be smoke or full")

In [ ]:
import subprocess
import sys

if UPLOADED_PROJECT_DIR is not None:
    PROJECT_DIR = Path(UPLOADED_PROJECT_DIR)
else:
    if SOURCE_DIR.exists():
        subprocess.run(["git", "-C", str(SOURCE_DIR), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(SOURCE_DIR)], check=True)
    PROJECT_DIR = SOURCE_DIR / "đồ án"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")], check=True)
sys.path.insert(0, str(PROJECT_DIR))

import json
import math
import os
import pickle
import random
import shutil
import threading
from concurrent.futures import ThreadPoolExecutor
from copy import deepcopy
from queue import Empty, Queue

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from data import TextStore, load_video_fold

if not DATA_ROOT.is_dir() and (DATA_ROOT.parent / "data.zip").is_file():
    DATA_ROOT = DATA_ROOT.parent / "data.zip"
if not DATA_ROOT.exists():
    raise FileNotFoundError(DATA_ROOT)
if not torch.cuda.is_available():
    raise RuntimeError("Hãy bật GPU trong Kaggle Settings")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VISIBLE_GPUS = torch.cuda.device_count()
WORKER_COUNT = min(GPU_WORKERS, VISIBLE_GPUS, len(FOLDS))
print("Data:", DATA_ROOT)
for gpu in range(VISIBLE_GPUS):
    print(f"GPU {gpu}: {torch.cuda.get_device_name(gpu)}")
print("Parallel workers:", WORKER_COUNT)

## Mô hình temporal RVQ-VAE và downstream classifiers

In [ ]:
VERSION = "temporal-rvq-grammar-v3"
CODEBOOK_SIZE = 64
QUANTIZERS = 4
LATENT_DIM = 128
LATENT_STEPS = 63
GRAMMAR_ORDER = 4
GRAMMAR_BINS = (1, 2, 4)
GRAMMAR_MAX_FEATURES = 40000
RVQ_BATCH_SIZE = 32
CLASSIFIER_BATCH_SIZE = 64

class EMACodebook(nn.Module):
    def __init__(self, size, dimension, decay=0.99, epsilon=1e-5):
        super().__init__()
        self.size = size
        self.decay = decay
        self.epsilon = epsilon
        initial = torch.randn(size, dimension) / math.sqrt(dimension)
        self.register_buffer("embedding", initial)
        self.register_buffer("cluster_size", torch.zeros(size))
        self.register_buffer("embedding_sum", initial.clone())

    def forward(self, values):
        flat = values.reshape(-1, values.shape[-1])
        search_values = flat.to(self.embedding.dtype)
        with torch.autocast(device_type=values.device.type, enabled=False):
            distances = search_values.pow(2).sum(1, keepdim=True) + self.embedding.pow(2).sum(1) - 2 * search_values @ self.embedding.t()
        indices = distances.argmin(1)
        quantized = F.embedding(indices, self.embedding).reshape_as(values)
        if self.training:
            ema_values = search_values.detach()
            one_hot = F.one_hot(indices, self.size).to(self.embedding.dtype)
            counts = one_hot.sum(0)
            sums = one_hot.t() @ ema_values
            self.cluster_size.mul_(self.decay).add_(counts, alpha=1 - self.decay)
            self.embedding_sum.mul_(self.decay).add_(sums, alpha=1 - self.decay)
            total = self.cluster_size.sum()
            smoothed = (self.cluster_size + self.epsilon) / (total + self.size * self.epsilon) * total
            self.embedding.copy_(self.embedding_sum / smoothed.unsqueeze(1).clamp_min(self.epsilon))
            dead = self.cluster_size < 0.1
            if dead.any() and len(flat):
                samples = ema_values[torch.randint(len(ema_values), (int(dead.sum()),), device=ema_values.device)]
                self.embedding[dead] = samples
                self.embedding_sum[dead] = samples
                self.cluster_size[dead] = 1.0
        commitment = F.mse_loss(values.float(), quantized.detach())
        return quantized.to(values.dtype), indices.reshape(values.shape[:-1]), commitment

class ResidualEMAQuantizer(nn.Module):
    def __init__(self, levels, size, dimension, commitment_weight=0.25):
        super().__init__()
        self.commitment_weight = commitment_weight
        self.codebooks = nn.ModuleList([EMACodebook(size, dimension) for _ in range(levels)])

    def forward(self, values):
        residual = values
        quantized_sum = torch.zeros_like(values)
        token_levels = []
        commitment = values.new_tensor(0.0)
        for codebook in self.codebooks:
            quantized, indices, level_loss = codebook(residual)
            quantized_sum = quantized_sum + quantized
            residual = residual - quantized.detach()
            token_levels.append(indices)
            commitment = commitment + level_loss
        straight_through = values + (quantized_sum - values).detach()
        return straight_through, self.commitment_weight * commitment, torch.stack(token_levels, dim=-1)

class TemporalRVQVAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(60, 128, 5, padding=2), nn.BatchNorm1d(128), nn.GELU(),
            nn.Conv1d(128, 192, 4, stride=2, padding=1), nn.BatchNorm1d(192), nn.GELU(),
            nn.Conv1d(192, 256, 3, stride=2, padding=1), nn.BatchNorm1d(256), nn.GELU(),
            nn.Conv1d(256, LATENT_DIM, 3, padding=1)
        )
        self.quantizer = ResidualEMAQuantizer(QUANTIZERS, CODEBOOK_SIZE, LATENT_DIM)
        self.decoder = nn.Sequential(
            nn.Conv1d(LATENT_DIM, 256, 3, padding=1), nn.BatchNorm1d(256), nn.GELU(),
            nn.ConvTranspose1d(256, 192, 3, stride=2, padding=1), nn.BatchNorm1d(192), nn.GELU(),
            nn.ConvTranspose1d(192, 128, 4, stride=2, padding=1), nn.BatchNorm1d(128), nn.GELU(),
            nn.Conv1d(128, 60, 5, padding=2)
        )

    def encode(self, values):
        return self.encoder(values.transpose(1, 2)).transpose(1, 2)

    def forward(self, values):
        latent = self.encode(values)
        quantized, commitment, tokens = self.quantizer(latent)
        reconstruction = self.decoder(quantized.transpose(1, 2)).transpose(1, 2)
        return reconstruction, commitment, tokens

    @torch.no_grad()
    def tokens(self, values):
        return self.quantizer(self.encode(values))[2]

class MotionTokenEmbedding(nn.Module):
    def __init__(self, codebooks, steps=LATENT_STEPS):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding.from_pretrained(weight.clone(), freeze=True) for weight in codebooks])
        self.position = nn.Parameter(torch.zeros(1, steps, LATENT_DIM))
        nn.init.normal_(self.position, std=0.02)
        self.norm = nn.LayerNorm(LATENT_DIM)
        self.dropout = nn.Dropout(0.2)

    def forward(self, tokens):
        values = torch.stack([embedding(tokens[:, :, level]) for level, embedding in enumerate(self.embeddings)]).sum(0)
        return self.dropout(self.norm(values + self.position[:, :values.shape[1]]))

class ResidualTemporalBlock(nn.Module):
    def __init__(self, dilation):
        super().__init__()
        self.layers = nn.Sequential(nn.Conv1d(128, 128, 3, padding=dilation, dilation=dilation), nn.BatchNorm1d(128), nn.GELU(), nn.Dropout(0.25), nn.Conv1d(128, 128, 3, padding=dilation, dilation=dilation), nn.BatchNorm1d(128))

    def forward(self, values):
        return F.gelu(values + self.layers(values))

class TokenCNN(nn.Module):
    def __init__(self, codebooks):
        super().__init__()
        self.embedding = MotionTokenEmbedding(codebooks)
        self.blocks = nn.Sequential(ResidualTemporalBlock(1), ResidualTemporalBlock(2), ResidualTemporalBlock(4), ResidualTemporalBlock(8))
        self.output = nn.Sequential(nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.35), nn.Linear(128, 1))

    def forward(self, tokens):
        values = self.blocks(self.embedding(tokens).transpose(1, 2))
        return self.output(torch.cat([values.mean(-1), values.amax(-1)], dim=1)).squeeze(1)

class TokenGRU(nn.Module):
    def __init__(self, codebooks):
        super().__init__()
        self.embedding = MotionTokenEmbedding(codebooks)
        self.gru = nn.GRU(LATENT_DIM, 96, num_layers=2, batch_first=True, bidirectional=True, dropout=0.25)
        self.attention = nn.Linear(192, 1)
        self.output = nn.Sequential(nn.Linear(384, 128), nn.GELU(), nn.Dropout(0.35), nn.Linear(128, 1))

    def forward(self, tokens):
        values, _ = self.gru(self.embedding(tokens))
        attention = torch.softmax(self.attention(values).squeeze(-1), dim=1).unsqueeze(-1)
        return self.output(torch.cat([(values * attention).sum(1), values.amax(1)], dim=1)).squeeze(1)

In [ ]:
def seed_worker(seed, device):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.set_device(device)
    torch.cuda.manual_seed(seed)

def array_loader(values, batch_size, shuffle, seed):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(TensorDataset(torch.as_tensor(values, dtype=torch.float32)), batch_size=batch_size, shuffle=shuffle, generator=generator, pin_memory=True)

def token_loader(tokens, labels, shuffle, seed):
    generator = torch.Generator().manual_seed(seed)
    dataset = TensorDataset(torch.as_tensor(tokens, dtype=torch.long), torch.as_tensor(labels, dtype=torch.float32))
    return DataLoader(dataset, batch_size=CLASSIFIER_BATCH_SIZE, shuffle=shuffle, generator=generator, pin_memory=True)

def cpu_state(model):
    return {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

def rvq_loss(model, batch):
    reconstruction, commitment, _ = model(batch)
    position = F.mse_loss(reconstruction[:, :, :30], batch[:, :, :30])
    velocity = F.mse_loss(reconstruction[:, :, 30:], batch[:, :, 30:])
    return position + velocity + commitment, position, velocity

def train_rvq(model, fit_values, val_values, device, seed):
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-5)
    scaler = torch.cuda.amp.GradScaler()
    fit_loader = array_loader(fit_values, RVQ_BATCH_SIZE, True, seed)
    val_loader = array_loader(val_values, RVQ_BATCH_SIZE, False, seed)
    best_loss = float("inf")
    best_state = cpu_state(model)
    stale = 0
    history = []
    for epoch in range(1, RVQ_EPOCHS + 1):
        model.train()
        train_total = 0.0
        train_count = 0
        for (batch,) in fit_loader:
            batch = batch.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast():
                loss, _, _ = rvq_loss(model, batch)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            train_total += float(loss.detach()) * len(batch)
            train_count += len(batch)
        model.eval()
        val_total = 0.0
        val_count = 0
        with torch.no_grad():
            for (batch,) in val_loader:
                batch = batch.to(device, non_blocking=True)
                with torch.cuda.amp.autocast():
                    loss, _, _ = rvq_loss(model, batch)
                val_total += float(loss) * len(batch)
                val_count += len(batch)
        train_loss = train_total / train_count
        val_loss = val_total / val_count
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
        if epoch == 1 or epoch % 10 == 0:
            print(f"RVQ epoch={epoch} train={train_loss:.5f} val={val_loss:.5f}", flush=True)
        if val_loss < best_loss - 1e-5:
            best_loss = val_loss
            best_state = cpu_state(model)
            stale = 0
        else:
            stale += 1
            if stale >= RVQ_PATIENCE:
                break
    model.load_state_dict(best_state)
    return history, best_loss

@torch.no_grad()
def encode_tokens(model, values, device):
    model.eval()
    batches = []
    for (batch,) in array_loader(values, 128, False, 0):
        batches.append(model.tokens(batch.to(device, non_blocking=True)).cpu().numpy())
    return np.concatenate(batches)

def metrics(labels, probabilities, threshold):
    labels = np.asarray(labels, dtype=np.int64)
    predictions = (np.asarray(probabilities) >= threshold).astype(np.int64)
    tn = int(np.sum((labels == 0) & (predictions == 0)))
    fp = int(np.sum((labels == 0) & (predictions == 1)))
    fn = int(np.sum((labels == 1) & (predictions == 0)))
    tp = int(np.sum((labels == 1) & (predictions == 1)))
    sensitivity = tp / (tp + fn) if tp + fn else float("nan")
    specificity = tn / (tn + fp) if tn + fp else float("nan")
    return {"sensitivity": sensitivity, "specificity": specificity, "balanced_accuracy": (sensitivity + specificity) / 2, "tn": tn, "fp": fp, "fn": fn, "tp": tp}, predictions

def best_threshold(labels, probabilities):
    candidates = np.linspace(0.05, 0.95, 181)
    scores = np.asarray([metrics(labels, probabilities, threshold)[0]["balanced_accuracy"] for threshold in candidates])
    best = np.flatnonzero(np.isclose(scores, scores.max()))
    index = int(best[np.argmin(np.abs(candidates[best] - 0.5))])
    return float(candidates[index]), float(scores[index])

@torch.no_grad()
def model_probabilities(model, tokens, device):
    model.eval()
    loader = DataLoader(TensorDataset(torch.as_tensor(tokens, dtype=torch.long)), batch_size=128, pin_memory=True)
    output = []
    for (batch,) in loader:
        output.append(torch.sigmoid(model(batch.to(device, non_blocking=True))).cpu().numpy())
    return np.concatenate(output)

def train_token_model(name, codebooks, fit_tokens, fit_labels, val_tokens, val_labels, test_tokens, test_labels, device, seed):
    model = (TokenCNN(codebooks) if name == "rvq_token_cnn" else TokenGRU(codebooks)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    negatives = float(np.sum(np.asarray(fit_labels) == 0))
    positives = float(np.sum(np.asarray(fit_labels) == 1))
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(negatives / positives, device=device))
    loader = token_loader(fit_tokens, fit_labels, True, seed)
    scaler = torch.cuda.amp.GradScaler()
    best_score = -1.0
    best_state = cpu_state(model)
    selected_threshold = 0.5
    selected_epoch = 0
    stale = 0
    for epoch in range(1, CLASSIFIER_EPOCHS + 1):
        model.train()
        for tokens, labels in loader:
            tokens = tokens.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast():
                loss = criterion(model(tokens), labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        val_probabilities = model_probabilities(model, val_tokens, device)
        threshold, score = best_threshold(val_labels, val_probabilities)
        if score > best_score + 1e-6:
            best_score = score
            best_state = cpu_state(model)
            selected_threshold = threshold
            selected_epoch = epoch
            stale = 0
        else:
            stale += 1
            if stale >= CLASSIFIER_PATIENCE:
                break
    model.load_state_dict(best_state)
    probabilities = model_probabilities(model, test_tokens, device)
    result, predictions = metrics(test_labels, probabilities, selected_threshold)
    return model, result, predictions, probabilities, selected_threshold, selected_epoch, best_score

## Huấn luyện 9 fold trên tối đa hai GPU

In [ ]:
initialization_lock = threading.Lock()

def codebook_statistics(tokens):
    rows = []
    for level in range(tokens.shape[-1]):
        counts = np.bincount(tokens[:, :, level].reshape(-1), minlength=CODEBOOK_SIZE).astype(float)
        probabilities = counts[counts > 0] / counts.sum()
        rows.append({"level": level + 1, "used_codes": int(np.count_nonzero(counts)), "perplexity": float(np.exp(-np.sum(probabilities * np.log(probabilities))))})
    return rows

def grammar_documents(tokens):
    documents = []
    for sample in tokens:
        words = []
        steps, levels = sample.shape
        for level in range(levels):
            sequence = [f"l{level}c{int(sample[step, level])}" for step in range(steps)]
            for order in range(1, GRAMMAR_ORDER + 1):
                words.extend(f"l{level}g{order}_" + "_".join(sequence[start:start + order]) for start in range(steps - order + 1))
        joint = ["j" + "_".join(str(int(code)) for code in sample[step]) for step in range(steps)]
        for divisions in GRAMMAR_BINS:
            boundaries = np.linspace(0, steps, divisions + 1, dtype=int)
            for part in range(divisions):
                segment = joint[boundaries[part]:boundaries[part + 1]]
                for order in range(1, GRAMMAR_ORDER + 1):
                    words.extend(f"p{divisions}b{part}g{order}_" + "__".join(segment[start:start + order]) for start in range(len(segment) - order + 1))
        documents.append(" ".join(words))
    return documents

def sklearn_row(name, classifier, vectorizer, fit_tokens, fit_labels, val_tokens, val_labels, test_tokens, test_labels, fold):
    fit_docs = grammar_documents(fit_tokens)
    val_docs = grammar_documents(val_tokens)
    test_docs = grammar_documents(test_tokens)
    fit_matrix = vectorizer.fit_transform(fit_docs)
    val_matrix = vectorizer.transform(val_docs)
    test_matrix = vectorizer.transform(test_docs)
    classifier.fit(fit_matrix, fit_labels)
    val_probabilities = classifier.predict_proba(val_matrix)[:, 1]
    threshold, val_score = best_threshold(val_labels, val_probabilities)
    probabilities = classifier.predict_proba(test_matrix)[:, 1]
    result, predictions = metrics(test_labels, probabilities, threshold)
    row = {"fold": fold, "classifier": name, "train_samples": len(fit_labels), "test_samples": len(test_labels), "vocabulary_size": len(vectorizer.vocabulary_), **result}
    return row, predictions, probabilities, threshold, val_score

def train_fold(fold, gpu):
    fold_dir = OUTPUT_DIR / f"fold_{fold}"
    fold_dir.mkdir(parents=True, exist_ok=True)
    cache_path = fold_dir / "result.json"
    if cache_path.is_file():
        cached = json.loads(cache_path.read_text())
        if cached.get("version") == VERSION and len(cached.get("rows", [])) == 4:
            print(f"GPU {gpu} fold {fold} skipped", flush=True)
            return cached["rows"]
    device = torch.device(f"cuda:{gpu}")
    seed_worker(SEED + fold, device)
    data = load_video_fold(TextStore(DATA_ROOT), fold)
    with initialization_lock:
        torch.manual_seed(SEED + fold)
        rvq = TemporalRVQVAE()
    rvq.to(device)
    history, best_reconstruction = train_rvq(rvq, data.x_fit, data.x_val, device, SEED + fold)
    train_tokens = encode_tokens(rvq, data.x_train, device)
    test_tokens = encode_tokens(rvq, data.x_test, device)
    split = len(data.y_fit)
    fit_tokens = train_tokens[:split]
    val_tokens = train_tokens[split:]
    codebooks = [codebook.embedding.detach().cpu().clone() for codebook in rvq.quantizer.codebooks]
    torch.save({"version": VERSION, "state_dict": cpu_state(rvq), "best_validation_loss": best_reconstruction}, fold_dir / "temporal_rvqvae.pt")
    pd.DataFrame(history).to_csv(fold_dir / "rvq_history.csv", index=False)
    (fold_dir / "codebook_statistics.json").write_text(json.dumps(codebook_statistics(train_tokens), indent=2))
    rows = []
    vectorizer = TfidfVectorizer(lowercase=False, token_pattern=r"[^ ]+", sublinear_tf=True, norm="l2", min_df=2, max_features=GRAMMAR_MAX_FEATURES, dtype=np.float32)
    logistic = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED + fold)
    row, predictions, probabilities, threshold, val_score = sklearn_row("rvq_grammar_logistic", logistic, vectorizer, fit_tokens, data.y_fit, val_tokens, data.y_val, test_tokens, data.y_test, fold)
    rows.append(row)
    with (fold_dir / "grammar_logistic.pkl").open("wb") as stream:
        pickle.dump({"vectorizer": vectorizer, "classifier": logistic, "threshold": threshold, "grammar_order": GRAMMAR_ORDER, "grammar_bins": GRAMMAR_BINS}, stream)
    pd.DataFrame({"y_true": data.y_test, "probability": probabilities, "y_pred": predictions}).to_csv(fold_dir / "rvq_grammar_logistic_predictions.csv", index=False)
    vectorizer = TfidfVectorizer(lowercase=False, token_pattern=r"[^ ]+", sublinear_tf=True, norm="l2", min_df=2, max_features=GRAMMAR_MAX_FEATURES, dtype=np.float32)
    mlp = MLPClassifier(hidden_layer_sizes=(256, 64), max_iter=1000, early_stopping=True, validation_fraction=0.125, n_iter_no_change=25, random_state=SEED + fold)
    row, predictions, probabilities, threshold, val_score = sklearn_row("rvq_grammar_mlp", mlp, vectorizer, fit_tokens, data.y_fit, val_tokens, data.y_val, test_tokens, data.y_test, fold)
    rows.append(row)
    pd.DataFrame({"y_true": data.y_test, "probability": probabilities, "y_pred": predictions}).to_csv(fold_dir / "rvq_grammar_mlp_predictions.csv", index=False)
    for offset, name in enumerate(("rvq_token_cnn", "rvq_token_gru")):
        model, result, predictions, probabilities, threshold, epoch, val_score = train_token_model(name, codebooks, fit_tokens, data.y_fit, val_tokens, data.y_val, test_tokens, data.y_test, device, SEED + fold * 10 + offset)
        row = {"fold": fold, "classifier": name, "train_samples": len(data.y_fit), "test_samples": len(data.y_test), "vocabulary_size": CODEBOOK_SIZE * QUANTIZERS, **result}
        rows.append(row)
        torch.save({"state_dict": cpu_state(model), "threshold": threshold, "best_epoch": epoch, "validation_balanced_accuracy": val_score}, fold_dir / f"{name}.pt")
        pd.DataFrame({"y_true": data.y_test, "probability": probabilities, "y_pred": predictions}).to_csv(fold_dir / f"{name}_predictions.csv", index=False)
        print(f"GPU {gpu} fold={fold} {name} BA={result['balanced_accuracy']:.4f}", flush=True)
        del model
        torch.cuda.empty_cache()
    payload = {"version": VERSION, "rows": rows}
    cache_path.write_text(json.dumps(payload, indent=2))
    del rvq
    torch.cuda.empty_cache()
    return rows

fold_queue = Queue()
for fold in FOLDS:
    fold_queue.put(fold)

def gpu_worker(gpu):
    rows = []
    while True:
        try:
            fold = fold_queue.get_nowait()
        except Empty:
            return rows
        print(f"GPU {gpu} started fold {fold}", flush=True)
        rows.extend(train_fold(fold, gpu))
        fold_queue.task_done()

with ThreadPoolExecutor(max_workers=WORKER_COUNT) as executor:
    futures = [executor.submit(gpu_worker, gpu) for gpu in range(WORKER_COUNT)]
    result_rows = [row for future in futures for row in future.result()]

fold_results = pd.DataFrame(result_rows).sort_values(["classifier", "fold"])
fold_results.to_csv(OUTPUT_DIR / "fold_results.csv", index=False)

## Bảng tổng hợp và so sánh

In [ ]:
def summarize(frame):
    rows = []
    for classifier, group in frame.groupby("classifier"):
        row = {"classifier": classifier, "n_folds": len(group)}
        for metric_name in ("sensitivity", "specificity", "balanced_accuracy"):
            values = group[metric_name].to_numpy(float)
            mean = float(values.mean())
            margin = float(1.96 * values.std(ddof=1) / np.sqrt(len(values))) if len(values) > 1 else float("nan")
            row[metric_name + "_mean"] = mean
            row[metric_name + "_ci_low"] = mean - margin
            row[metric_name + "_ci_high"] = mean + margin
        row["retrained_cnn_video_ba"] = 0.89402
        row["published_cnn_video_ba"] = 0.9066064667
        row["delta_vs_retrained"] = row["balanced_accuracy_mean"] - 0.89402
        rows.append(row)
    return pd.DataFrame(rows)

summary = summarize(fold_results)
summary.to_csv(OUTPUT_DIR / "summary.csv", index=False)
display(fold_results.round(4))
display(summary.round(4))

positions = np.arange(len(summary))
values = 100 * summary["balanced_accuracy_mean"].to_numpy()
errors = 100 * (summary["balanced_accuracy_ci_high"] - summary["balanced_accuracy_mean"]).to_numpy()
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(positions, values, yerr=errors if np.isfinite(errors).all() else None, capsize=5)
ax.axhline(89.402, color="tab:red", linestyle=":", label="Retrained video CNN: 89.40%")
ax.axhline(90.6606467, color="black", linestyle="--", label="Published video CNN: 90.66%")
ax.set_xticks(positions, summary["classifier"], rotation=15)
ax.set_ylabel("Balanced accuracy (%)")
ax.set_ylim(0, 100)
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "comparison.png", dpi=180)
plt.show()
archive = shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR)
print("Artifacts:", archive)